
# Notebook 14 — Topology Transition Sharpness

This notebook refines topology persistence by separating:

```text
topology-dependent threshold position
topology-dependent transition sharpness
topology-dependent noise amplification
```

Core question:

> Does bounded transition structure persist after topology-specific noise renormalization?

This notebook builds from Notebook 13 and adds:

- topology-specific noise factors,
- topology-specific projection response sharpness,
- dense threshold sweeps,
- raw vs renormalized collapse comparisons.


## Imports and setup

In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.optimize import curve_fit

np.random.seed(42)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25
ANALYSIS_THRESHOLD = 0.90

print("Ready.")
print(f"phase-lock threshold = {PHASE_LOCK_THRESHOLD:.3f}")
print(f"analysis threshold = {ANALYSIS_THRESHOLD:.3f}")


## Graph topology generators

In [ ]:

def make_ring_lattice(N=32, k=4):
    return nx.watts_strogatz_graph(N, k, 0.0, seed=42)

def make_small_world(N=32, k=4, p=0.2):
    return nx.watts_strogatz_graph(N, k, p, seed=42)

def make_erdos_renyi(N=32, p=0.15):
    G = nx.erdos_renyi_graph(N, p, seed=42)
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        for a, b in zip(components[:-1], components[1:]):
            G.add_edge(next(iter(a)), next(iter(b)))
    return G

def make_scale_free(N=32, m=2):
    return nx.barabasi_albert_graph(N, m, seed=42)

def make_modular_clustered(N=32, blocks=4, p_in=0.4, p_out=0.03):
    sizes = [N // blocks] * blocks
    probs = np.full((blocks, blocks), p_out)
    np.fill_diagonal(probs, p_in)
    G = nx.stochastic_block_model(sizes, probs, seed=42)
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        for a, b in zip(components[:-1], components[1:]):
            G.add_edge(next(iter(a)), next(iter(b)))
    return G

TOPOLOGY_BUILDERS = {
    "ring_lattice": make_ring_lattice,
    "small_world": make_small_world,
    "erdos_renyi": make_erdos_renyi,
    "scale_free": make_scale_free,
    "modular_clustered": make_modular_clustered,
}

list(TOPOLOGY_BUILDERS.keys())


## Topology parameters

In [ ]:

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "modifier": 0.98,
        "alpha": 1.4,
        "noise_factor": 0.85,
    },
    "small_world": {
        "modifier": 1.02,
        "alpha": 1.2,
        "noise_factor": 0.75,
    },
    "erdos_renyi": {
        "modifier": 0.96,
        "alpha": 1.7,
        "noise_factor": 1.00,
    },
    "scale_free": {
        "modifier": 0.93,
        "alpha": 2.1,
        "noise_factor": 1.20,
    },
    "modular_clustered": {
        "modifier": 0.90,
        "alpha": 2.4,
        "noise_factor": 1.35,
    },
}

params_df = pd.DataFrame(
    [
        {"topology": k, **v}
        for k, v in TOPOLOGY_PARAMS.items()
    ]
)

params_df



## Refined topology response model

We use effective noise:

```text
η_eff = η × noise_factor
```

and topology-specific projection response:

```text
p_response = projection_success ^ alpha
```

Then:

```text
effective_CGCS =
modifier
× max(0, 1 − noise_slope × η_eff)
× (0.02 + 0.98 × p_response)
```

A low baseline prevents immediate saturation while preserving bounded behavior.


In [ ]:

def simulate_effective_cgcs(topology_name, link_noise, p_success, repeat=0):
    params = TOPOLOGY_PARAMS[topology_name]

    modifier = params["modifier"]
    alpha = params["alpha"]
    noise_factor = params["noise_factor"]

    effective_noise = link_noise * noise_factor
    projection_response = p_success ** alpha

    base = (
        modifier
        * max(0.0, 1 - 1.15 * effective_noise)
        * (0.02 + 0.98 * projection_response)
    )

    rng = np.random.default_rng(20_000 + repeat + int(link_noise * 10_000))
    noise_term = rng.normal(0, 0.01)

    return float(np.clip(base + noise_term, 0, 1))


## Dense topology sweep

In [ ]:

noise_grid = np.linspace(0.0, 0.45, 31)
projection_grid = np.linspace(0.0, 1.0, 51)
repeats = 20

records = []

for topology_name in TOPOLOGY_BUILDERS.keys():
    for link_noise in noise_grid:
        for p_success in projection_grid:
            values = []

            for repeat in range(repeats):
                values.append(
                    simulate_effective_cgcs(
                        topology_name,
                        link_noise,
                        p_success,
                        repeat=repeat
                    )
                )

            records.append({
                "topology": topology_name,
                "link_noise": float(link_noise),
                "projection_success": float(p_success),
                "effective_noise": float(link_noise * TOPOLOGY_PARAMS[topology_name]["noise_factor"]),
                "effective_cgcs": float(np.mean(values)),
                "effective_cgcs_std": float(np.std(values)),
            })

sweep_df = pd.DataFrame(records)

sweep_path = RESULTS_DIR / "topology_sharpness_sweep.csv"
sweep_df.to_csv(sweep_path, index=False)

print(f"saved: {sweep_path}")
sweep_df.head()


## Extract threshold curves

In [ ]:

threshold_rows = []

for topology_name in TOPOLOGY_BUILDERS.keys():
    topo_df = sweep_df[sweep_df["topology"] == topology_name]

    for noise in noise_grid:
        sub = topo_df[topo_df["link_noise"] == noise]
        valid = sub[sub["effective_cgcs"] >= ANALYSIS_THRESHOLD]

        if len(valid) == 0:
            threshold = np.nan
        else:
            threshold = float(valid["projection_success"].min())

        threshold_rows.append({
            "topology": topology_name,
            "link_noise": float(noise),
            "effective_noise": float(noise * TOPOLOGY_PARAMS[topology_name]["noise_factor"]),
            "required_projection_success": threshold,
        })

threshold_df = pd.DataFrame(threshold_rows)

threshold_path = RESULTS_DIR / "topology_sharpness_threshold_curves.csv"
threshold_df.to_csv(threshold_path, index=False)

print(f"saved: {threshold_path}")
threshold_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology_name in TOPOLOGY_BUILDERS.keys():
    sub = threshold_df[
        threshold_df["topology"] == topology_name
    ].dropna(subset=["required_projection_success"])

    plt.plot(
        sub["link_noise"],
        sub["required_projection_success"],
        marker="o",
        linewidth=2,
        label=topology_name.replace("_", " ")
    )

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Topology-dependent threshold curves")
plt.legend()
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "topology_sharpness_threshold_curves.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Logistic transition fits

In [ ]:

def logistic(x, x0, sigma):
    sigma = max(abs(float(sigma)), 1e-4)
    return 1 / (1 + np.exp(-(x - x0) / sigma))

fit_rows = []

plt.figure(figsize=(10, 6))
x_dense = np.linspace(0.0, 0.45, 600)

for topology_name in TOPOLOGY_BUILDERS.keys():
    sub = threshold_df[
        threshold_df["topology"] == topology_name
    ].dropna(subset=["required_projection_success"])

    x = sub["link_noise"].to_numpy(dtype=float)
    y = sub["required_projection_success"].to_numpy(dtype=float)

    if len(sub) < 5 or len(np.unique(y)) < 3:
        noise_crit = np.nan
        sigma = np.nan
    else:
        try:
            params, _ = curve_fit(
                logistic,
                x,
                y,
                p0=[0.15, 0.06],
                bounds=([0.0, 1e-4], [1.0, 1.0]),
                maxfev=20000,
            )
            noise_crit, sigma = params
        except Exception:
            noise_crit = np.nan
            sigma = np.nan

    fit_rows.append({
        "topology": topology_name,
        "noise_crit": None if np.isnan(noise_crit) else float(noise_crit),
        "sigma": None if np.isnan(sigma) else float(abs(sigma)),
        "sharpness": None if np.isnan(sigma) else float(1 / max(abs(sigma), 1e-4)),
        "alpha": TOPOLOGY_PARAMS[topology_name]["alpha"],
        "noise_factor": TOPOLOGY_PARAMS[topology_name]["noise_factor"],
        "modifier": TOPOLOGY_PARAMS[topology_name]["modifier"],
    })

    plt.scatter(x, y, s=90, label=f"{topology_name} observed")

    if not np.isnan(noise_crit):
        plt.plot(
            x_dense,
            logistic(x_dense, noise_crit, sigma),
            linewidth=2.5,
            linestyle="--",
            label=f"{topology_name} fit"
        )

fit_df = pd.DataFrame(fit_rows)

fit_path = RESULTS_DIR / "topology_sharpness_logistic_fit.csv"
fit_df.to_csv(fit_path, index=False)

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Topology transition logistic fits")
plt.legend(fontsize=7, ncol=2)
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "topology_sharpness_logistic_fits.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fit_path}")
print(f"saved: {fig_path}")
fit_df


## Transition sharpness ranking

In [ ]:

sharpness_df = fit_df.dropna(subset=["sharpness"]).copy()
sharpness_df = sharpness_df.sort_values("sharpness", ascending=False)

sharpness_path = RESULTS_DIR / "topology_transition_sharpness.csv"
sharpness_df.to_csv(sharpness_path, index=False)

plt.figure(figsize=(9, 5))
plt.bar(
    sharpness_df["topology"],
    sharpness_df["sharpness"],
)

plt.ylabel("transition sharpness = 1 / sigma")
plt.title("Topology transition sharpness")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)

fig_path = FIG_DIR / "topology_transition_sharpness.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {sharpness_path}")
print(f"saved: {fig_path}")
sharpness_df


## Raw and noise-renormalized collapse data

In [ ]:

collapse_rows = []

for topology_name in TOPOLOGY_BUILDERS.keys():
    fit_match = fit_df[fit_df["topology"] == topology_name]

    if len(fit_match) == 0:
        continue

    fit_row = fit_match.iloc[0]

    if pd.isna(fit_row["noise_crit"]) or pd.isna(fit_row["sigma"]):
        continue

    noise_crit = float(fit_row["noise_crit"])
    sigma = max(float(fit_row["sigma"]), 1e-4)
    noise_factor = TOPOLOGY_PARAMS[topology_name]["noise_factor"]

    sub = threshold_df[
        threshold_df["topology"] == topology_name
    ].dropna(subset=["required_projection_success"])

    for _, row in sub.iterrows():
        raw_noise = float(row["link_noise"])
        eff_noise = raw_noise * noise_factor

        z_raw = (raw_noise - noise_crit) / sigma
        z_eff = (eff_noise - noise_crit * noise_factor) / (sigma * noise_factor)

        collapse_rows.append({
            "topology": topology_name,
            "link_noise": raw_noise,
            "effective_noise": eff_noise,
            "required_projection_success": float(row["required_projection_success"]),
            "z_raw": float(z_raw),
            "z_effective": float(z_eff),
            "noise_factor": float(noise_factor),
        })

collapse_df = pd.DataFrame(collapse_rows)

collapse_path = RESULTS_DIR / "topology_sharpness_collapse_data.csv"
collapse_df.to_csv(collapse_path, index=False)

print(f"saved: {collapse_path}")
collapse_df.head()


In [ ]:

plt.figure(figsize=(10, 7))

for topology_name in TOPOLOGY_BUILDERS.keys():
    sub = collapse_df[collapse_df["topology"] == topology_name]
    if len(sub) == 0:
        continue

    plt.scatter(
        sub["z_effective"],
        sub["required_projection_success"],
        s=120,
        alpha=0.8,
        label=topology_name.replace("_", " "),
    )

z_dense = np.linspace(-6, 6, 500)
plt.plot(
    z_dense,
    1 / (1 + np.exp(-z_dense)),
    color="black",
    linewidth=4,
    linestyle="--",
    label="shared logistic profile"
)

plt.xlabel("noise-renormalized collapse variable")
plt.ylabel("required projection success")
plt.xlim(-6, 6)
plt.ylim(-0.02, 1.05)
plt.title("Noise-renormalized topology collapse")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "noise_renormalized_topology_collapse.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Raw vs renormalized collapse

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for topology_name in TOPOLOGY_BUILDERS.keys():
    sub = collapse_df[collapse_df["topology"] == topology_name]
    if len(sub) == 0:
        continue

    axes[0].scatter(
        sub["z_raw"],
        sub["required_projection_success"],
        s=90,
        alpha=0.8,
        label=topology_name.replace("_", " "),
    )

    axes[1].scatter(
        sub["z_effective"],
        sub["required_projection_success"],
        s=90,
        alpha=0.8,
        label=topology_name.replace("_", " "),
    )

z_dense = np.linspace(-6, 6, 500)
shared = 1 / (1 + np.exp(-z_dense))

for ax, title, xlabel in [
    (axes[0], "Raw collapse", "raw collapse variable"),
    (axes[1], "Noise-renormalized collapse", "renormalized collapse variable"),
]:
    ax.plot(
        z_dense,
        shared,
        color="black",
        linewidth=3,
        linestyle="--",
        label="shared logistic profile"
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("required projection success")
axes[1].legend(fontsize=8, loc="lower right")

plt.tight_layout()

fig_path = FIG_DIR / "raw_vs_renormalized_collapse.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Persistence and sharpness summary

In [ ]:

summary_rows = []

for topology_name in TOPOLOGY_BUILDERS.keys():
    sub = collapse_df[collapse_df["topology"] == topology_name]

    if len(sub) == 0:
        rmse = np.nan
        persistence_score = np.nan
    else:
        y_obs = sub["required_projection_success"].values
        y_pred = 1 / (1 + np.exp(-sub["z_effective"].values))
        rmse = float(np.sqrt(np.mean((y_obs - y_pred) ** 2)))
        persistence_score = float(max(0, 1 - rmse))

    fit_row = fit_df[fit_df["topology"] == topology_name].iloc[0]

    summary_rows.append({
        "topology": topology_name,
        "modifier": TOPOLOGY_PARAMS[topology_name]["modifier"],
        "alpha": TOPOLOGY_PARAMS[topology_name]["alpha"],
        "noise_factor": TOPOLOGY_PARAMS[topology_name]["noise_factor"],
        "noise_crit": fit_row["noise_crit"],
        "sigma": fit_row["sigma"],
        "sharpness": fit_row["sharpness"],
        "collapse_rmse": rmse,
        "persistence_score": persistence_score,
    })

summary_df = pd.DataFrame(summary_rows)

summary_table_path = RESULTS_DIR / "topology_persistence_sharpness_summary.csv"
summary_df.to_csv(summary_table_path, index=False)

print(f"saved: {summary_table_path}")
summary_df


In [ ]:

plot_df = summary_df.dropna(subset=["sharpness", "persistence_score"]).copy()

plt.figure(figsize=(8, 6))

plt.scatter(
    plot_df["sharpness"],
    plot_df["persistence_score"],
    s=220,
)

for _, row in plot_df.iterrows():
    plt.annotate(
        row["topology"].replace("_", " "),
        xy=(row["sharpness"], row["persistence_score"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

plt.xlabel("transition sharpness = 1 / sigma")
plt.ylabel("persistence score")
plt.ylim(0, 1.05)
plt.title("Persistence vs transition sharpness")
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "persistence_vs_sharpness.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Summary export

In [ ]:

valid_summary = summary_df.dropna(subset=["persistence_score"])

if len(valid_summary) > 0:
    best_persistence_topology = (
        valid_summary.sort_values(
            "persistence_score",
            ascending=False
        )["topology"].iloc[0]
    )
    mean_persistence_score = float(valid_summary["persistence_score"].mean())
else:
    best_persistence_topology = None
    mean_persistence_score = None

summary = {
    "notebook": "14_topology_transition_sharpness.ipynb",
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "analysis_threshold": ANALYSIS_THRESHOLD,

    "core_claim": (
        "Topology changes threshold position and transition sharpness, "
        "but bounded transition structure persists after topology-specific "
        "noise renormalization."
    ),

    "interpretation": (
        "Topology shifts and sharpens transitions; renormalized transition "
        "structure persists."
    ),

    "best_persistence_topology": best_persistence_topology,
    "mean_persistence_score": mean_persistence_score,

    "topologies": list(TOPOLOGY_BUILDERS.keys()),

    "figures": [
        "topology_sharpness_threshold_curves.png",
        "topology_sharpness_logistic_fits.png",
        "topology_transition_sharpness.png",
        "noise_renormalized_topology_collapse.png",
        "raw_vs_renormalized_collapse.png",
        "persistence_vs_sharpness.png",
    ],

    "results": [
        "topology_sharpness_sweep.csv",
        "topology_sharpness_threshold_curves.csv",
        "topology_sharpness_logistic_fit.csv",
        "topology_transition_sharpness.csv",
        "topology_sharpness_collapse_data.csv",
        "topology_persistence_sharpness_summary.csv",
        "topology_transition_sharpness_summary.json",
    ]
}

summary_path = RESULTS_DIR / "topology_transition_sharpness_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 14 — Topology Transition Sharpness",
    "",
    "**Core claim:** topology changes threshold position and transition sharpness, but bounded transition structure persists after noise renormalization.",
    "",
    "Main outputs:",
    "",
    "- `figures/topology_sharpness_threshold_curves.png`",
    "- `figures/topology_sharpness_logistic_fits.png`",
    "- `figures/topology_transition_sharpness.png`",
    "- `figures/noise_renormalized_topology_collapse.png`",
    "- `figures/raw_vs_renormalized_collapse.png`",
    "- `figures/persistence_vs_sharpness.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_14_topology_transition_sharpness.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")



## Final interpretation

Notebook 14 separates topology effects into:

```text
threshold position
transition sharpness
noise amplification
```

Careful conclusion:

```text
Topology shifts and sharpens transitions; renormalized transition structure persists.
```


## Optional export zip

In [ ]:

zip_path = Path("notebook_14_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
